In [2]:
# ==============================================================================
# Cell 1: Setup, Paths, and NLP Dependencies (Lexical-Categorical Space)
# ==============================================================================

"""
Configuración inicial para el procesamiento NLP y la construcción del espacio
Léxico-Categorial. Inicializa el modelo de spaCy y define las rutas.
"""

import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import tgt
import spacy

# ------------------------------------------------------------------------------
# Rutas de sistema
# ------------------------------------------------------------------------------
BASE_DIR = Path("/home/amont21/Documentos/voxelwise modeling/ds003020/derivatives/TextGrids")

# ------------------------------------------------------------------------------
# Configuración del Modelo NLP (spaCy)
# ------------------------------------------------------------------------------
# Se recomienda "en_core_web_sm" para pruebas rápidas y "en_core_web_trf" 
# (Transformers) para la extracción de las categorías gramaticales definitivas.
SPACY_MODEL_NAME = "en_core_web_trf"

print(f"Cargando modelo de spaCy: {SPACY_MODEL_NAME}...")
try:
    nlp = spacy.load(SPACY_MODEL_NAME)
    print("✅ Modelo cargado exitosamente.")
except OSError:
    print(f"❌ Error: El modelo '{SPACY_MODEL_NAME}' no está instalado.")
    print(f"Instálalo con: python -m spacy download {SPACY_MODEL_NAME}")
    
# ------------------------------------------------------------------------------
# Parámetros temporales
# ------------------------------------------------------------------------------
HIGH_RES_FS = 100  # Resolución de 100 Hz (10 ms)

Cargando modelo de spaCy: en_core_web_trf...
✅ Modelo cargado exitosamente.


In [3]:
# ==============================================================================
# Cell 2: Acoustic Audit of Pauses and Metalinguistic Tags
# ==============================================================================

"""
Script de auditoría para analizar la duración empírica de las pausas y ruidos.
Extrae la duración en segundos de cada intervalo vacío o etiqueta metalingüística 
para permitir al investigador definir umbrales informados empíricamente al 
asignar signos de puntuación (comas vs. puntos) para el análisis sintáctico.
"""

def audit_pause_durations(base_dir: Path) -> None:
    """
    Recorre los TextGrids y almacena la duración en segundos de los silencios
    y las etiquetas metalingüísticas, generando un reporte estadístico.
    
    Args:
        base_dir (Path): Ruta al directorio de TextGrids.
    """
    textgrid_files = list(base_dir.rglob("*.TextGrid"))
    
    # Usaremos un diccionario de listas para guardar todas las duraciones por etiqueta
    duration_data = defaultdict(list)
    
    for file_path in textgrid_files:
        try:
            tg = tgt.io.read_textgrid(str(file_path), include_empty_intervals=True)
            word_tier = None
            
            for tier_name in tg.get_tier_names():
                if 'word' in tier_name.lower():
                    word_tier = tg.get_tier_by_name(tier_name)
                    break
                    
            if word_tier is not None:
                for interval in word_tier.intervals:
                    token = interval.text.strip().lower()
                    duration = interval.end_time - interval.start_time
                    
                    if token == "":
                        token = "<empty_silence>"
                        
                    # Filtrar solo lo que consideramos ruido o pausas (según auditoría previa)
                    if token == "<empty_silence>" or re.search(r'\[|\]|\{|\}|\<|\>|spn|^sp$|^sil$|^br$|^lg$', token):
                        duration_data[token].append(duration)
                        
        except Exception:
            pass

    # Transformar a DataFrame para hacer estadística descriptiva (EDA)
    stats_list = []
    for tag, durations in duration_data.items():
        arr = np.array(durations)
        stats_list.append({
            'Tag': tag,
            'Count': len(arr),
            'Mean_sec': np.mean(arr),
            'Min_sec': np.min(arr),
            '25%_sec': np.percentile(arr, 25),
            'Median_sec': np.median(arr),
            '75%_sec': np.percentile(arr, 75),
            'Max_sec': np.max(arr)
        })
        
    df_stats = pd.DataFrame(stats_list)
    
    # Ordenar por frecuencia (Count)
    if not df_stats.empty:
        df_stats = df_stats.sort_values(by='Count', ascending=False).reset_index(drop=True)
        
        print("-" * 70)
        print("AUDITORÍA DE DURACIÓN DE PAUSAS Y ETIQUETAS METALINGÜÍSTICAS")
        print("Valores expresados en segundos.")
        print("-" * 70)
        
        # Opciones para mostrar números bonitos
        pd.set_option('display.float_format', lambda x: '%.3f' % x)
        display(df_stats)
        pd.reset_option('display.float_format')
        
    else:
        print("No se encontraron etiquetas de pausa/ruido.")

import re
audit_pause_durations(BASE_DIR)

----------------------------------------------------------------------
AUDITORÍA DE DURACIÓN DE PAUSAS Y ETIQUETAS METALINGÜÍSTICAS
Valores expresados en segundos.
----------------------------------------------------------------------


,Tag,Count,Mean_sec,Min_sec,25%_sec,Median_sec,75%_sec,Max_sec
0,sp,23432,0.487,0.003,0.140,0.329,0.649,11.554
1,{br},1828,0.347,0.030,0.226,0.319,0.439,1.674
2,{lg},1381,1.343,0.030,0.564,0.988,1.716,9.478
3,{ns},940,1.305,0.040,0.429,0.815,1.445,17.081
4,{ls},271,0.131,0.030,0.080,0.110,0.154,0.469
5,<empty_silence>,95,0.107,0.002,0.012,0.012,0.045,1.346
6,{cg},31,0.374,0.030,0.290,0.351,0.485,0.749
7,{sp},4,0.290,0.096,0.177,0.291,0.404,0.481
8,{ns,2,0.482,0.449,0.465,0.482,0.498,0.514
9,{ig},2,0.668,0.519,0.594,0.668,0.743,0.818


In [7]:
# ==============================================================================
# Cell 3: Punctuation Simulator and Text Reconstruction
# ==============================================================================

"""
Reconstruye la historia como un texto continuo con sintaxis simulada.
Utiliza umbrales de duración empíricos (basados en la auditoría acústica) para 
insertar signos de puntuación de manera paramétrica. Además, genera un mapa de 
alineación (char offsets) para mapear los resultados de spaCy de vuelta a 
los tiempos exactos de la fMRI.
"""

from typing import Tuple, List, Dict, Union

# ------------------------------------------------------------------------------
# Parámetros empíricos de puntuación (Basados en auditoría)
# ------------------------------------------------------------------------------
# Si la pausa es >= 0.5s, se asume fin de enunciado (punto).
THRESHOLD_PERIOD_SEC = 0.50  

# Si la pausa es >= 0.2s y < 0.5s, se asume pausa intonacional (coma).
THRESHOLD_COMMA_SEC = 0.20  

# Expresión regular para detectar todo el inventario de ruidos auditado
NOISE_PATTERN = re.compile(r'\[|\]|\{|\}|\<|\>|spn|^sp$|^sil$|^br$|^lg$', re.IGNORECASE)

def reconstruct_and_map_text(textgrid_path: Union[str, Path]) -> Tuple[str, pd.DataFrame]:
    """
    Convierte el TextGrid en un string puntado y crea una tabla de alineación.
    
    Args:
        textgrid_path (Path o str): Ruta al archivo TextGrid.
        
    Returns:
        Tuple[str, pd.DataFrame]: 
            - El texto completo continuo con puntuación simulada.
            - Un DataFrame con los tiempos originales y su posición en caracteres.
    """
    tg = tgt.io.read_textgrid(str(textgrid_path), include_empty_intervals=True)
    
    word_tier = None
    for name in tg.get_tier_names():
        if 'word' in name.lower():
            word_tier = tg.get_tier_by_name(name)
            break
            
    if word_tier is None:
        raise ValueError(f"No se encontró capa 'words' en {textgrid_path}")
        
    reconstructed_text = ""
    word_mapping = []
    
    for interval in word_tier.intervals:
        token = interval.text.strip()
        is_empty = token == ""
        is_noise = bool(NOISE_PATTERN.search(token))
        
        # 1. Si es un ruido o silencio, evaluamos su duración para la puntuación
        if is_empty or is_noise:
            duration = interval.end_time - interval.start_time
            
            if duration >= THRESHOLD_PERIOD_SEC:
                # Evitar poner múltiples puntos seguidos
                if not reconstructed_text.endswith(". "):
                    # Si terminaba en coma, la reemplazamos por punto
                    if reconstructed_text.endswith(", "):
                        reconstructed_text = reconstructed_text[:-2]
                    reconstructed_text += ". "
                    
            elif duration >= THRESHOLD_COMMA_SEC:
                if not (reconstructed_text.endswith(". ") or reconstructed_text.endswith(", ")):
                    reconstructed_text += ", "
            
            # Si es < THRESHOLD_COMMA_SEC, no hacemos nada (micropausa)
            continue
            
        # 2. Si es una palabra real, la limpiamos y la añadimos al texto
        clean_word = re.sub(r'[^a-zA-Z\']', '', token).lower()
        if not clean_word:
            continue
            
        # Calcular los índices de caracteres (dónde empieza y termina esta palabra en el string final)
        char_start = len(reconstructed_text)
        char_end = char_start + len(clean_word)
        
        # Añadir al mapa de alineación
        word_mapping.append({
            'original_word': clean_word,
            'start_time': interval.start_time,
            'end_time': interval.end_time,
            'char_start': char_start,
            'char_end': char_end
        })
        
        # Añadir la palabra al texto continuo seguida de un espacio
        reconstructed_text += clean_word + " "
        
    return reconstructed_text.strip(), pd.DataFrame(word_mapping)

# ==============================================================================
# Prueba Visual del Reconstructor
# ==============================================================================
try:
    textgrid_files = list(BASE_DIR.rglob("*.TextGrid"))
    valid_file = next((f for f in textgrid_files if f.name not in ['legacy.TextGrid', 'exorcism.TextGrid']), None)
            
    if valid_file:
        print(f"Reconstruyendo texto para: {valid_file.name}")
        full_text, df_map = reconstruct_and_map_text(valid_file)
        
        print("\n--- FRAGMENTO DEL TEXTO SIMULADO (Primeros 500 caracteres) ---")
        print(full_text[:500] + "...")
        print("-" * 62)
        
        print("\n--- MAPA DE ALINEACIÓN (Primeras 5 palabras detectadas) ---")
        display(df_map.head())
        
    else:
        print("No se encontraron archivos válidos.")

except Exception as e:
    print(f"Error: {e}")

Reconstruyendo texto para: odetostepfather.TextGrid

--- FRAGMENTO DEL TEXTO SIMULADO (Primeros 500 caracteres) ---
under the influence is our topic tonight and uh i i thought of a lot of different things and , and the one i kinda wanted to talk about was , my , secret , influence um . there are people in our lives that are , incredibly important to us and , i would even m mention to say some of the most important people everybody aside from my mother in my life doesn't know anything about this person , who i think about every day , and his name was michael marquis so , if this were a violin piece it would be...
--------------------------------------------------------------

--- MAPA DE ALINEACIÓN (Primeras 5 palabras detectadas) ---


,original_word,start_time,end_time,char_start,char_end
0,under,0.032426,0.221995,0,5
1,the,0.221995,0.321769,6,9
2,influence,0.321769,0.870522,10,19
3,is,0.870522,1.000227,20,22
4,our,1.000227,1.109977,23,26


In [9]:
# ==============================================================================
# Cell 3: Advanced Punctuation Simulator and Text Reconstruction
# ==============================================================================

"""
Reconstruye la historia como un texto continuo con sintaxis simulada.

JUSTIFICACIÓN METODOLÓGICA DE UMBRALES (Basada en auditoría acústica):
- THRESHOLD_COMMA_SEC (0.35s): La mediana empírica de pausas cortas ('sp') 
  en el corpus es de ~0.33s. Umbrales menores (ej. 0.20s) capturan vacilaciones 
  articulatorias (hesitations) que insertan comas dentro de sintagmas nominales 
  (ej. "my , secret"), destruyendo la estructura de dependencias para el NLP. 
  Se fija en 0.35s para aislar pausas intencionales de nivel clausal.
- THRESHOLD_PERIOD_SEC (0.65s): El 75% de las pausas 'sp' caen debajo de 0.65s.
  Pausas superiores a este umbral (como las risas 'lg' con mediana ~0.98s) 
  indican consistentemente el final de un enunciado/idea completa.
  
Además, se aplica un post-procesamiento ortográfico para asegurar que los signos 
de puntuación no tengan espacios precedentes (ej. "word ," -> "word,"), 
optimizando la tokenización por sub-palabras del modelo Transformer de spaCy.
"""

import re
import random
from typing import Tuple, List
import pandas as pd
import tgt
from pathlib import Path

# ------------------------------------------------------------------------------
# Parámetros empíricos de puntuación
# ------------------------------------------------------------------------------
THRESHOLD_PERIOD_SEC = 0.65  
THRESHOLD_COMMA_SEC = 0.35   

NOISE_PATTERN = re.compile(r'\[|\]|\{|\}|\<|\>|spn|^sp$|^sil$|^br$|^lg$', re.IGNORECASE)

def reconstruct_and_map_text(textgrid_path: Path) -> Tuple[str, pd.DataFrame]:
    """
    Convierte el TextGrid en un string puntado ortográficamente correcto 
    y crea una tabla de alineación de caracteres a tiempo (Piedra Rosetta).
    """
    tg = tgt.io.read_textgrid(str(textgrid_path), include_empty_intervals=True)
    
    word_tier = None
    for name in tg.get_tier_names():
        if 'word' in name.lower():
            word_tier = tg.get_tier_by_name(name)
            break
            
    if word_tier is None:
        raise ValueError(f"No se encontró capa 'words' en {textgrid_path}")
        
    raw_tokens = []
    word_mapping = []
    
    for interval in word_tier.intervals:
        token = interval.text.strip()
        is_empty = token == ""
        is_noise = bool(NOISE_PATTERN.search(token))
        
        # 1. Procesamiento de pausas
        if is_empty or is_noise:
            duration = interval.end_time - interval.start_time
            
            if duration >= THRESHOLD_PERIOD_SEC:
                # Evitar múltiples puntos
                if raw_tokens and raw_tokens[-1] not in ['.', ',']:
                    raw_tokens.append('.')
                elif raw_tokens and raw_tokens[-1] == ',':
                    raw_tokens[-1] = '.' # Escalar coma a punto
                    
            elif duration >= THRESHOLD_COMMA_SEC:
                if raw_tokens and raw_tokens[-1] not in ['.', ',']:
                    raw_tokens.append(',')
            continue
            
        # 2. Procesamiento de palabras reales
        clean_word = re.sub(r'[^a-zA-Z\']', '', token).lower()
        if not clean_word:
            continue
            
        raw_tokens.append(clean_word)
        
        # Guardamos la metadata temporal temporalmente (los índices de char se calculan después)
        word_mapping.append({
            'original_word': clean_word,
            'start_time': interval.start_time,
            'end_time': interval.end_time
        })
        
    # 3. Ensamblaje Ortográfico Correcto
    # Unimos todo con espacios y luego corregimos la puntuación
    reconstructed_text = " ".join(raw_tokens)
    
    # Expresiones regulares para arreglar "word ." -> "word." y "word ," -> "word,"
    reconstructed_text = re.sub(r'\s+([.,])', r'\1', reconstructed_text)
    
    # 4. Cálculo final del mapa de alineación (Character Offsets)
    # Buscamos dónde quedó cada palabra en el texto final
    current_search_idx = 0
    final_mapping = []
    
    for word_info in word_mapping:
        word = word_info['original_word']
        # Buscar la palabra a partir del índice actual para mantener el orden
        match = re.search(r'\b' + re.escape(word) + r'\b', reconstructed_text[current_search_idx:])
        
        if match:
            char_start = current_search_idx + match.start()
            char_end = current_search_idx + match.end()
            
            final_mapping.append({
                'original_word': word,
                'start_time': word_info['start_time'],
                'end_time': word_info['end_time'],
                'char_start': char_start,
                'char_end': char_end
            })
            # Actualizar el índice de búsqueda
            current_search_idx = char_end
            
    return reconstructed_text.strip(), pd.DataFrame(final_mapping)

# ==============================================================================
# Prueba Visual Aleatoria (Validación Metodológica)
# ==============================================================================
try:
    textgrid_files = list(BASE_DIR.rglob("*.TextGrid"))
    valid_files = [f for f in textgrid_files if f.name not in ['legacy.TextGrid', 'exorcism.TextGrid']]
    
    if len(valid_files) >= 3:
        # Seleccionar 3 historias al azar
        sample_files = random.sample(valid_files, 3)
        
        print("🔍 REVISIÓN METODOLÓGICA DE LA SINTAXIS SIMULADA\n" + "="*50)
        
        for file in sample_files:
            text, _ = reconstruct_and_map_text(file)
            print(f"\n📖 Historia: {file.name}")
            # Mostrar los primeros 500 caracteres
            print(f'"{text[:500]}..."')
            
    else:
        print("No hay suficientes archivos válidos para la muestra.")

except Exception as e:
    print(f"Error: {e}")

🔍 REVISIÓN METODOLÓGICA DE LA SINTAXIS SIMULADA

📖 Historia: comingofageondeathrow.TextGrid
"when i was fifteen. there were two things that made up a good day. the first was chocolate chip waffles for breakfast. and the second was a chess tournament after school. i was a good if apathetic student. i had a penchant for bad puns i was kind of nerdy and awkward wore these baggy shirts that my mom bought me and uh i had this long shaggy hair that kinda made me look like, an indian anakin skywalker if you can imagine that. wasn't a good look. but yeah that was me at fifteen and. i was fiftee..."

📖 Historia: kiksuya.TextGrid
"so i'm at this party back home in minneapolis minnesota. the the evening's winding down and i'm gonna try to sneak out when this lady stops me. uh she just shakes my hand and, introduces herself and says that my uncle told her to come talk to me, now this uncle is not my uncle in the sense of blood relation he's my uncle indian way one of the many men who helped, raise

In [10]:
# ==============================================================================
# Cell 4: Lexical-Categorical Extraction Engine (NLP POS Tagging)
# ==============================================================================

"""
Genera la matriz temporal de características léxico-categoriales.

JUSTIFICACIÓN METODOLÓGICA (Exclusión de Verbos Finitos):
Para evitar la colinealidad con el fenómeno principal de investigación (el 
Tiempo Gramatical Finito), este espacio de control categorial extrae las 
clases de palabras (sustantivos, adjetivos, etc.) pero EXCLUYE intencionalmente 
los verbos finitos (pasado 'VBD' y presente 'VBP'/'VBZ'). 

En su lugar, se incluye únicamente la categoría de verbos no finitos 
('non_finite_verb' -> infinitivos, gerundios, participios) para controlar 
la presencia de material verbal general en la oración sin absorber la varianza 
exclusiva del contraste flexivo finito.
"""

from typing import Union
import numpy as np

# Definición del Espacio de Características Categoriales
POS_FEATURE_NAMES = [
    'noun',              # Sustantivos y Nombres propios (NOUN, PROPN)
    'adjective',         # Adjetivos (ADJ)
    'adverb',            # Adverbios (ADV)
    'pronoun',           # Pronombres (PRON)
    'preposition',       # Preposiciones y Posposiciones (ADP)
    'conjunction',       # Conjunciones (CCONJ, SCONJ)
    'determiner',        # Determinantes (DET)
    'non_finite_verb'    # Verbos No Finitos (VB, VBG, VBN)
]

def classify_spacy_token(spacy_token: spacy.tokens.Token) -> list:
    """
    Clasifica un token de spaCy en el espacio de características del estudio.
    Aplica el filtro estricto de finitud verbal basado en el Penn Treebank.
    
    Args:
        spacy_token: Objeto Token analizado por spaCy.
        
    Returns:
        list: Vector binario (0s y 1s) de las 8 categorías.
    """
    features = [0] * len(POS_FEATURE_NAMES)
    
    pos = spacy_token.pos_
    tag = spacy_token.tag_
    
    if pos in ['NOUN', 'PROPN']:
        features[0] = 1
    elif pos == 'ADJ':
        features[1] = 1
    elif pos == 'ADV':
        features[2] = 1
    elif pos == 'PRON':
        features[3] = 1
    elif pos == 'ADP':
        features[4] = 1
    elif pos in ['CCONJ', 'SCONJ']:
        features[5] = 1
    elif pos == 'DET':
        features[6] = 1
    elif pos in ['VERB', 'AUX']:
        # Filtro estricto usando etiquetas del Penn Treebank (tag_):
        # VB  = Base form (infinitive)
        # VBG = Gerund or present participle
        # VBN = Past participle
        # Si es VBD, VBP, o VBZ, es finito y se excluye (puros ceros).
        if tag in ['VB', 'VBG', 'VBN']:
            features[7] = 1
            
    return features

def extract_categorical_space(df_alignment: pd.DataFrame, full_text: str, nlp_model: spacy.language.Language, fs: int) -> pd.DataFrame:
    """
    Realiza el POS Tagging sobre el texto continuo y alinea los resultados a 100 Hz.
    
    Args:
        df_alignment (pd.DataFrame): Mapa temporal generado en la Celda 3.
        full_text (str): Texto con puntuación simulada.
        nlp_model: Modelo cargado de spaCy.
        fs (int): Frecuencia de muestreo.
        
    Returns:
        pd.DataFrame: Matriz categorial (tipo int8 para eficiencia de memoria).
    """
    # 1. NLP Inference: spaCy analiza todo el contexto
    doc = nlp_model(full_text)
    
    # 2. Inverted Mapping (Character -> spaCy Token)
    # Fundamental porque spaCy puede dividir "don't" en "do" y "n't", afectando 
    # la alineación directa por palabra.
    char_to_token = {}
    for token in doc:
        for i in range(token.idx, token.idx + len(token.text)):
            char_to_token[i] = token
            
    # 3. Preparación de Matriz (100 Hz)
    max_time = df_alignment['end_time'].max() if not df_alignment.empty else 0
    total_samples = int(np.ceil(max_time * fs))
    feature_matrix = np.zeros((total_samples, len(POS_FEATURE_NAMES)), dtype=np.int8)
    
    # 4. Proyección Temporal
    for _, row in df_alignment.iterrows():
        # Usar el punto central de la palabra para capturar el token de spaCy correspondiente
        mid_char = (row['char_start'] + row['char_end']) // 2
        spacy_token = char_to_token.get(mid_char)
        
        if spacy_token:
            pos_vector = classify_spacy_token(spacy_token)
            
            if sum(pos_vector) > 0:
                start_idx = int(np.floor(row['start_time'] * fs))
                end_idx = int(np.ceil(row['end_time'] * fs))
                end_idx = min(end_idx, total_samples)
                
                feature_matrix[start_idx:end_idx, :] = pos_vector
                
    time_axis = np.arange(total_samples) / fs
    df_features = pd.DataFrame(feature_matrix, columns=POS_FEATURE_NAMES, index=time_axis)
    df_features.index.name = 'time_seconds'
    
    return df_features

In [11]:
# ==============================================================================
# Cell 5: Execution and Lexical-Categorical Visualization
# ==============================================================================

"""
Ejecuta la extracción NLP, alinea con el fMRI y muestra el espacio de control 
categorial (matriz binaria) resultante.
"""

try:
    if 'valid_file' in locals():
        print(f"Extrayendo categorías gramaticales para: {valid_file.name}")
        
        # Obtenemos el texto y alineación (Celda 3)
        full_text, df_map = reconstruct_and_map_text(valid_file)
        
        # Generamos el espacio (Celda 4)
        df_categorical_space = extract_categorical_space(df_map, full_text, nlp, HIGH_RES_FS)
        
        print("\nInformación del Espacio Léxico-Categorial:")
        print(f"  -> Dimensiones (Muestras x Rasgos): {df_categorical_space.shape}")
        print(f"  -> Tipo de dato: {df_categorical_space.values.dtype}\n")
        
        print("Visualización de momentos donde hay actividad categorial (cambios):")
        
        # Filtramos para mostrar los instantes donde la red detectó una categoría 
        active_cat_samples = df_categorical_space[df_categorical_space.sum(axis=1) > 0]
        
        # Eliminamos filas consecutivas repetidas para ver la transición entre palabras
        display(active_cat_samples.drop_duplicates().head(15))
        
    else:
        print("Asegúrate de tener un archivo válido seleccionado.")

except Exception as e:
    print(f"Ocurrió un error en la extracción categorial: {str(e)}")

Extrayendo categorías gramaticales para: odetostepfather.TextGrid

Información del Espacio Léxico-Categorial:
  -> Dimensiones (Muestras x Rasgos): (73039, 8)
  -> Tipo de dato: int8

Visualización de momentos donde hay actividad categorial (cambios):


,noun,adjective,adverb,pronoun,preposition,conjunction,determiner,non_finite_verb
time_seconds,,,,,,,,
0.03,0,0,0,0,1,0,0,0
0.22,0,0,0,0,0,0,1,0
0.32,1,0,0,0,0,0,0,0
1.00,0,0,0,1,0,0,0,0
1.89,0,0,0,0,0,1,0,0
3.16,0,1,0,0,0,0,0,0
5.31,0,0,1,0,0,0,0,0
6.24,0,0,0,0,0,0,0,1
